## ERP115928

**paper:** [PMID: 32184981](https://pmc.ncbi.nlm.nih.gov/articles/PMC7069322/) - New genome assembly of the barn owl (Tyto alba alba), 2020

**date, curator:** 2026-03-24, Sara Carsanaro

**notes**
* manually updated the infoOrgan, was able to confirm tissues in methods section
* male, 57-day old nestling - nestling stage is neonate stage in uberon

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,kidney,UBERON:0002113,kidney,perfect match
1,heart,UBERON:0000948,heart,perfect match
2,liver,UBERON:0002107,liver,perfect match
3,testis,UBERON:0000473,testis,perfect match
4,growing back feathers,UBERON:0018539,dorsal feather,other
5,thalamus,UBERON:0001897,dorsal plus ventral thalamus,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,57_day_old nestling,UBERON:0007221,neonate stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "ERP115928"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 17/17 [00:19<00:00,  1.13s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,,,,kidney,57_day_old nestling,perfect match,not documented,,,,,507980,,,,,,24ki6830,SAMEA5733646,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,24ki6830,,,,,public,TRANSCRIPTOMIC,RANDOM
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,,,,heart,57_day_old nestling,perfect match,not documented,,,,,507980,,,,,,23He6830,SAMEA5733645,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,23He6830,,,,,public,TRANSCRIPTOMIC,RANDOM
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,,,,liver,57_day_old nestling,perfect match,not documented,,,,,507980,,,,,,21Li6830,SAMEA5733644,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,21Li6830,,,,,public,TRANSCRIPTOMIC,RANDOM
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,,,,testis,57_day_old nestling,perfect match,not documented,,,,,507980,,,,,,15Te6830,SAMEA5733643,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,15Te6830,,,,,public,TRANSCRIPTOMIC,RANDOM
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,,,,growing back feathers,57_day_old nestling,other,not documented,,,,,507980,,,,,,14FB6830,SAMEA5733642,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,14FB6830,,,,,public,TRANSCRIPTOMIC,RANDOM
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,,,,thalamus,57_day_old nestling,perfect match,not documented,,,,,507980,,,,,,10Th6830,SAMEA5733641,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,10Th6830,,,,,public,TRANSCRIPTOMIC,RANDOM


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['growing back feathers' 'heart' 'kidney' 'liver' 'testis' 'thalamus']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['57_day_old nestling']


In [7]:
# all
library.loc[:,'stageId'] = 'UBERON:0007221'
library.loc[:,'stageName'] = 'neonate stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'missing child term'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,,,,507980,,,,,,24ki6830,SAMEA5733646,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,24ki6830,,,,,public,TRANSCRIPTOMIC,RANDOM
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,,,,507980,,,,,,23He6830,SAMEA5733645,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,23He6830,,,,,public,TRANSCRIPTOMIC,RANDOM
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,,,,507980,,,,,,21Li6830,SAMEA5733644,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,21Li6830,,,,,public,TRANSCRIPTOMIC,RANDOM
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,,,,507980,,,,,,15Te6830,SAMEA5733643,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,15Te6830,,,,,public,TRANSCRIPTOMIC,RANDOM
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,,,,507980,,,,,,14FB6830,SAMEA5733642,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,14FB6830,,,,,public,TRANSCRIPTOMIC,RANDOM
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0007221,neonate stage,,thalamus,57_day_old nestling,perfect match,not documented,missing child term,,,,507980,,,,,,10Th6830,SAMEA5733641,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,10Th6830,,,,,public,TRANSCRIPTOMIC,RANDOM


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [8]:
library.loc[:,'sex'] = 'M'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,,,,,,24ki6830,SAMEA5733646,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,24ki6830,,,,,public,TRANSCRIPTOMIC,RANDOM
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,,,,,,23He6830,SAMEA5733645,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,23He6830,,,,,public,TRANSCRIPTOMIC,RANDOM
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,,,,,,21Li6830,SAMEA5733644,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,21Li6830,,,,,public,TRANSCRIPTOMIC,RANDOM
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,,,,,,15Te6830,SAMEA5733643,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,15Te6830,,,,,public,TRANSCRIPTOMIC,RANDOM
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,M,,,507980,,,,,,14FB6830,SAMEA5733642,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,14FB6830,,,,,public,TRANSCRIPTOMIC,RANDOM
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0007221,neonate stage,,thalamus,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,,,,,,10Th6830,SAMEA5733641,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,10Th6830,,,,,public,TRANSCRIPTOMIC,RANDOM


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'KAPA Stranded mRNA-Seq kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,24ki6830,SAMEA5733646,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,24ki6830,,,,,public,TRANSCRIPTOMIC,RANDOM
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,23He6830,SAMEA5733645,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,23He6830,,,,,public,TRANSCRIPTOMIC,RANDOM
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,21Li6830,SAMEA5733644,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,21Li6830,,,,,public,TRANSCRIPTOMIC,RANDOM
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,15Te6830,SAMEA5733643,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,15Te6830,,,,,public,TRANSCRIPTOMIC,RANDOM
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,14FB6830,SAMEA5733642,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,14FB6830,,,,,public,TRANSCRIPTOMIC,RANDOM
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0007221,neonate stage,,thalamus,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,10Th6830,SAMEA5733641,,,,,,,,,,,24/03/2026,KAPA stranded RNA,,10Th6830,,,,,public,TRANSCRIPTOMIC,RANDOM


#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [11]:
library.loc[:,'sampleAge_value'] = '57'
library.loc[:,'sampleAge_unit'] = 'day'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,24ki6830,SAMEA5733646,57,day,,,,,,,,,24/03/2026,KAPA stranded RNA,,24ki6830,,,,,public,TRANSCRIPTOMIC,RANDOM
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,23He6830,SAMEA5733645,57,day,,,,,,,,,24/03/2026,KAPA stranded RNA,,23He6830,,,,,public,TRANSCRIPTOMIC,RANDOM
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,21Li6830,SAMEA5733644,57,day,,,,,,,,,24/03/2026,KAPA stranded RNA,,21Li6830,,,,,public,TRANSCRIPTOMIC,RANDOM
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,15Te6830,SAMEA5733643,57,day,,,,,,,,,24/03/2026,KAPA stranded RNA,,15Te6830,,,,,public,TRANSCRIPTOMIC,RANDOM
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,14FB6830,SAMEA5733642,57,day,,,,,,,,,24/03/2026,KAPA stranded RNA,,14FB6830,,,,,public,TRANSCRIPTOMIC,RANDOM
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0007221,neonate stage,,thalamus,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,10Th6830,SAMEA5733641,57,day,,,,,,,,,24/03/2026,KAPA stranded RNA,,10Th6830,,,,,public,TRANSCRIPTOMIC,RANDOM


#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-24'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,24ki6830,SAMEA5733646,57,day,,,,,,,,SAC,2026-03-24,KAPA stranded RNA,,24ki6830,,,,,public,TRANSCRIPTOMIC,RANDOM
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,23He6830,SAMEA5733645,57,day,,,,,,,,SAC,2026-03-24,KAPA stranded RNA,,23He6830,,,,,public,TRANSCRIPTOMIC,RANDOM
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,21Li6830,SAMEA5733644,57,day,,,,,,,,SAC,2026-03-24,KAPA stranded RNA,,21Li6830,,,,,public,TRANSCRIPTOMIC,RANDOM
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,15Te6830,SAMEA5733643,57,day,,,,,,,,SAC,2026-03-24,KAPA stranded RNA,,15Te6830,,,,,public,TRANSCRIPTOMIC,RANDOM
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,14FB6830,SAMEA5733642,57,day,,,,,,,,SAC,2026-03-24,KAPA stranded RNA,,14FB6830,,,,,public,TRANSCRIPTOMIC,RANDOM
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0007221,neonate stage,,thalamus,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,10Th6830,SAMEA5733641,57,day,,,,,,,,SAC,2026-03-24,KAPA stranded RNA,,10Th6830,,,,,public,TRANSCRIPTOMIC,RANDOM


#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 32184981'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,24ki6830,SAMEA5733646,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
1,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,23He6830,SAMEA5733645,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
2,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,21Li6830,SAMEA5733644,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
3,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,15Te6830,SAMEA5733643,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
4,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,14FB6830,SAMEA5733642,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
5,ERX3418283,ERP115928,Illumina HiSeq 2500,ERS3536942,UBERON:0001897,dorsal plus ventral thalamus,UBERON:0007221,neonate stage,,thalamus,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,10Th6830,SAMEA5733641,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115928,High quality assembly and transcriptome of the European barn owl (Tyto alba alba).,"A high quality genome assembly of the European barn owl (Tyto alba alba) of 1.219 Gbp was assembled by mixing Illumina (117 x) and Pacific Biosciences (PacBio) long reads (12 x) technologies. BUSCO (Universal Single-Copy Orthologs) analysis revealed a high assembly completeness with only 1.8 % of the genes missing out of 4’915 avian orthologs searched and thus the European barn owl genome assembly appeared to be as good as the genomes of the zebra finch (Taeniopygia guttata) or the collared flycatcher (Ficedula albicollis). By mapping the reads of the female American barn owl (Tyto furcata pratincola) to the male European barn owl reads, several structural variants are detected and 70 Mbp of the Z chromosome is identified. The scaffold belonging to the Z chromosomes is further confirmed by mapping the European barn owl genome to the chromosomes of zebra finch. The completeness of the European barn owl genome is further demonstrated with a set of 128 proteins missing in the chicken genome, out of which 122 (95.3 %) were retrieved in the European barn owl transcripts and out of them reciprocal alignment recovered 94 input proteins (72.9 %). High quality genome is of key importance for future population genomics and comparative avian studies. This is exemplified by the finding that the barn owl position in the avian tree of life differs when the European or the American assembly is used.",SRA,,,,,,,PRJEB33158,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115928,High quality assembly and transcriptome of the European barn owl (Tyto alba alba).,"A high quality genome assembly of the European barn owl (Tyto alba alba) of 1.219 Gbp was assembled by mixing Illumina (117 x) and Pacific Biosciences (PacBio) long reads (12 x) technologies. BUSCO (Universal Single-Copy Orthologs) analysis revealed a high assembly completeness with only 1.8 % of the genes missing out of 4’915 avian orthologs searched and thus the European barn owl genome assembly appeared to be as good as the genomes of the zebra finch (Taeniopygia guttata) or the collared flycatcher (Ficedula albicollis). By mapping the reads of the female American barn owl (Tyto furcata pratincola) to the male European barn owl reads, several structural variants are detected and 70 Mbp of the Z chromosome is identified. The scaffold belonging to the Z chromosomes is further confirmed by mapping the European barn owl genome to the chromosomes of zebra finch. The completeness of the European barn owl genome is further demonstrated with a set of 128 proteins missing in the chicken genome, out of which 122 (95.3 %) were retrieved in the European barn owl transcripts and out of them reciprocal alignment recovered 94 input proteins (72.9 %). High quality genome is of key importance for future population genomics and comparative avian studies. This is exemplified by the finding that the barn owl position in the avian tree of life differs when the European or the American assembly is used.",SRA,partial,Bgee 1K,6,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB33158,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '32184981'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7069322/'
experiment.loc[:,'DOI'] = '10.1002/ece3.5991'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115928,High quality assembly and transcriptome of the European barn owl (Tyto alba alba).,"A high quality genome assembly of the European barn owl (Tyto alba alba) of 1.219 Gbp was assembled by mixing Illumina (117 x) and Pacific Biosciences (PacBio) long reads (12 x) technologies. BUSCO (Universal Single-Copy Orthologs) analysis revealed a high assembly completeness with only 1.8 % of the genes missing out of 4’915 avian orthologs searched and thus the European barn owl genome assembly appeared to be as good as the genomes of the zebra finch (Taeniopygia guttata) or the collared flycatcher (Ficedula albicollis). By mapping the reads of the female American barn owl (Tyto furcata pratincola) to the male European barn owl reads, several structural variants are detected and 70 Mbp of the Z chromosome is identified. The scaffold belonging to the Z chromosomes is further confirmed by mapping the European barn owl genome to the chromosomes of zebra finch. The completeness of the European barn owl genome is further demonstrated with a set of 128 proteins missing in the chicken genome, out of which 122 (95.3 %) were retrieved in the European barn owl transcripts and out of them reciprocal alignment recovered 94 input proteins (72.9 %). High quality genome is of key importance for future population genomics and comparative avian studies. This is exemplified by the finding that the barn owl position in the avian tree of life differs when the European or the American assembly is used.",SRA,partial,Bgee 1K,6,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB33158,32184981,https://pmc.ncbi.nlm.nih.gov/articles/PMC7069322/,10.1002/ece3.5991,,


#### comments

In [19]:
experiment.loc[:,'comment'] = 'removed DNA seq and PacBio libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115928,High quality assembly and transcriptome of the European barn owl (Tyto alba alba).,"A high quality genome assembly of the European barn owl (Tyto alba alba) of 1.219 Gbp was assembled by mixing Illumina (117 x) and Pacific Biosciences (PacBio) long reads (12 x) technologies. BUSCO (Universal Single-Copy Orthologs) analysis revealed a high assembly completeness with only 1.8 % of the genes missing out of 4’915 avian orthologs searched and thus the European barn owl genome assembly appeared to be as good as the genomes of the zebra finch (Taeniopygia guttata) or the collared flycatcher (Ficedula albicollis). By mapping the reads of the female American barn owl (Tyto furcata pratincola) to the male European barn owl reads, several structural variants are detected and 70 Mbp of the Z chromosome is identified. The scaffold belonging to the Z chromosomes is further confirmed by mapping the European barn owl genome to the chromosomes of zebra finch. The completeness of the European barn owl genome is further demonstrated with a set of 128 proteins missing in the chicken genome, out of which 122 (95.3 %) were retrieved in the European barn owl transcripts and out of them reciprocal alignment recovered 94 input proteins (72.9 %). High quality genome is of key importance for future population genomics and comparative avian studies. This is exemplified by the finding that the barn owl position in the avian tree of life differs when the European or the American assembly is used.",SRA,partial,Bgee 1K,6,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB33158,32184981,https://pmc.ncbi.nlm.nih.gov/articles/PMC7069322/,10.1002/ece3.5991,,removed DNA seq and PacBio libraries


#### save complete file

In [20]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [21]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
56621,SRX8571231,SRP267755,Illumina HiSeq 2500,SRS6863252,UBERON:0002435,striatum,UBERON:0000113,post-juvenile adult stage,,Striatum,adult,perfect match,not documented,perfect match,M,,,8996,,,polyA,,,YPT65112,SAMN15249105,,,,,,,PMID: 34009300,,,SAC,2026-03-24
56622,SRX8571220,SRP267755,Illumina HiSeq 2500,SRS6863241,UBERON:0002106,spleen,UBERON:0000113,post-juvenile adult stage,,Spleen,adult,perfect match,not documented,perfect match,M,,,8996,,,polyA,,,YPT65111,SAMN15249104,,,,,,,PMID: 34009300,,,SAC,2026-03-24
56623,ERX3418288,ERP115928,Illumina HiSeq 2500,ERS3536947,UBERON:0002113,kidney,UBERON:0007221,neonate stage,,kidney,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,24ki6830,SAMEA5733646,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
56624,ERX3418287,ERP115928,Illumina HiSeq 2500,ERS3536946,UBERON:0000948,heart,UBERON:0007221,neonate stage,,heart,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,23He6830,SAMEA5733645,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
56625,ERX3418286,ERP115928,Illumina HiSeq 2500,ERS3536945,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,21Li6830,SAMEA5733644,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
56626,ERX3418285,ERP115928,Illumina HiSeq 2500,ERS3536944,UBERON:0000473,testis,UBERON:0007221,neonate stage,,testis,57_day_old nestling,perfect match,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,15Te6830,SAMEA5733643,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24
56627,ERX3418284,ERP115928,Illumina HiSeq 2500,ERS3536943,UBERON:0018539,dorsal feather,UBERON:0007221,neonate stage,,growing back feathers,57_day_old nestling,other,not documented,missing child term,M,,,507980,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,14FB6830,SAMEA5733642,57,day,,,,,PMID: 32184981,,,SAC,2026-03-24


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1114,SRP445606,The copepod Eurytemora affinis as a relevant s...,"Compared to freshwater ecosystems, the health ...",SRA,total,Bgee 1K,15,Universal Plus mRNA-Seq Library Preparation Kit,full_length,,PRJNA986950,37660773,https://www.sciencedirect.com/science/article/...,10.1016/j.envpol.2023.122482,,ecotoxicological study
1115,SRP267755,Whole-genome re-sequencing for guinea fowl (Nu...,We sequenced a total of 129 samples from Afric...,SRA,partial,Bgee 1K,10,,,,PRJNA639777,34009300,https://pmc.ncbi.nlm.nih.gov/articles/PMC8214406/,10.1093/gbe/evab090,,"rejected majority of samples, DNA seq or PacBio"
1116,ERP115928,High quality assembly and transcriptome of the...,A high quality genome assembly of the European...,SRA,partial,Bgee 1K,6,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB33158,32184981,https://pmc.ncbi.nlm.nih.gov/articles/PMC7069322/,10.1002/ece3.5991,,removed DNA seq and PacBio libraries


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [29]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./
	../NCBI_output/

no changes added to commit (use "git add" and/or "git commit -a")


In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop 0ffafac] adding annotated bulk experiment ERP115928
 2 files changed, 7 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.61 KiB | 669.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   b8b2e18..0ffafac  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push